# 🧠 Email BERT Classifier — Google Colab Training
**Trains a multi-task DistilBERT model to:**
- ✅ Detect Spam vs Ham
- ✅ Classify Email Category (work / personal / promotions / spam)
- ✅ Score Priority (Low / Medium / High)

---
### ⚡ Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your 4 datasets to **Google Drive** at: `My Drive/email_datasets/`
   - `spam.csv`
   - `spam_assassin.csv`
   - `combined_data.csv`
   - `emails.csv`
3. Then **Run All** ▶️

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers datasets accelerate scikit-learn pandas

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DATASETS_DIR = '/content/drive/MyDrive/email_datasets'
MODELS_DIR   = '/content/drive/MyDrive/email_models'
os.makedirs(MODELS_DIR, exist_ok=True)

# Verify datasets exist
for f in ['spam.csv', 'spam_assassin.csv', 'combined_data.csv', 'emails.csv']:
    path = f'{DATASETS_DIR}/{f}'
    exists = os.path.exists(path)
    size = round(os.path.getsize(path)/1e6, 1) if exists else 0
    print(f'  [{"✅" if exists else "❌"}] {f}  ({size} MB)')

In [ ]:
# ── Cell 4: Text cleaning & labeling helpers ─────────────────────────────────
import re
import numpy as np
import pandas as pd

_HTML_RE  = re.compile(r'<[^>]+>')
_URL_RE   = re.compile(r'http\S+|www\.\S+')
_EMAIL_RE = re.compile(r'\S+@\S+')
_SPACE_RE = re.compile(r'\s+')

def clean_text(text):
    text = str(text)
    text = _HTML_RE.sub(' ', text)
    text = _URL_RE.sub(' URL ', text)
    text = _EMAIL_RE.sub(' EMAIL ', text)
    text = re.sub(r"[^a-zA-Z0-9\s.,!?']", ' ', text)
    text = _SPACE_RE.sub(' ', text).strip()
    return text[:512]

_URGENT = re.compile(
    r'\b(urgent|asap|immediately|deadline|action required|important|'
    r'confirm|verify|critical|alert|respond|due today)\b', re.I)
_PROMO = re.compile(
    r'\b(offer|sale|discount|free|win|prize|click|subscribe|unsubscribe|'
    r'limited time|deal|buy now|order now)\b', re.I)

CATEGORY_LABELS = {0: 'personal', 1: 'promotions', 2: 'spam', 3: 'work'}
PRIORITY_LABELS = {0: 'Low', 1: 'Medium', 2: 'High'}
SPAM_LABELS     = {0: 'ham', 1: 'spam'}

CATEGORY_TO_IDX = {v: k for k, v in CATEGORY_LABELS.items()}
PRIORITY_TO_IDX = {'Low': 0, 'Medium': 1, 'High': 2}

def assign_category(row):
    if row['label'] == 1:
        return 'spam'
    text = str(row['text']).lower()
    if _URGENT.search(text) or any(w in text for w in
            ('meeting', 'project', 'report', 'deadline', 'invoice',
             'client', 'budget', 'schedule', 'conference')):
        return 'work'
    if _PROMO.search(text) or any(w in text for w in
            ('offer', 'sale', 'discount', 'newsletter', 'promo')):
        return 'promotions'
    return 'personal'

def assign_priority(row):
    if row['label'] == 1:
        return 'Low'
    text = str(row['text'])
    if _URGENT.search(text):
        return 'High'
    if _PROMO.search(text):
        return 'Low'
    return 'Medium'

print('✅ Helper functions defined')

In [ ]:
# ── Cell 5: Load & merge all 4 datasets ──────────────────────────────────────
MAX_PER_CLASS = 40_000  # cap per spam/ham class

print('Loading spam.csv...')
df1 = pd.read_csv(f'{DATASETS_DIR}/spam.csv', encoding='latin-1')[['v1', 'v2']]
df1.columns = ['raw_label', 'text']
df1['label'] = (df1['raw_label'].str.lower().str.strip() == 'spam').astype(int)
df1.drop(columns=['raw_label'], inplace=True)
print(f'  spam.csv:         {len(df1):,} rows | spam={df1["label"].sum():,}')

print('Loading spam_assassin.csv...')
df2 = pd.read_csv(f'{DATASETS_DIR}/spam_assassin.csv', encoding='latin-1', on_bad_lines='skip')
text_col  = next((c for c in df2.columns if 'text' in c.lower()), df2.columns[0])
label_col = next((c for c in df2.columns if any(x in c.lower() for x in ['label','target','class'])), df2.columns[-1])
df2 = df2[[text_col, label_col]].copy()
df2.columns = ['text', 'label']
df2['label'] = df2['label'].apply(lambda x: 1 if str(x).lower() in ('spam','1') else 0)
print(f'  spam_assassin.csv:{len(df2):,} rows | spam={df2["label"].sum():,}')

print('Loading combined_data.csv...')
df3 = pd.read_csv(f'{DATASETS_DIR}/combined_data.csv', encoding='latin-1', on_bad_lines='skip')[['label','text']]
df3['label'] = pd.to_numeric(df3['label'], errors='coerce').fillna(0).astype(int).clip(0,1)
print(f'  combined_data.csv:{len(df3):,} rows | spam={df3["label"].sum():,}')

print('Loading emails.csv (Enron - 30k sample, all ham)...')
df4 = pd.read_csv(f'{DATASETS_DIR}/emails.csv', encoding='latin-1',
                  usecols=['message'], nrows=30_000, on_bad_lines='skip')
df4.rename(columns={'message': 'text'}, inplace=True)
df4['label'] = 0
df4['text'] = df4['text'].apply(lambda m: str(m).split('\n\n', 1)[1] if '\n\n' in str(m) else str(m))
print(f'  emails.csv:       {len(df4):,} rows | spam=0')

# Merge
df = pd.concat([df1, df2, df3, df4], ignore_index=True)
print(f'\nCombined raw: {len(df):,} rows')

# Clean
df['text'] = df['text'].apply(clean_text)
df = df[df['text'].str.len() > 10].copy()
df.drop_duplicates(subset=['text'], inplace=True)
print(f'After clean/dedup: {len(df):,} rows')

# Balance
ham_df  = df[df['label'] == 0].sample(min(MAX_PER_CLASS, len(df[df['label']==0])), random_state=42)
spam_df = df[df['label'] == 1].sample(min(MAX_PER_CLASS, len(df[df['label']==1])), random_state=42)
df = pd.concat([ham_df, spam_df], ignore_index=True)
print(f'Balanced: {len(df):,} rows  (ham={len(ham_df):,} | spam={len(spam_df):,})')

# Add category & priority labels
print('Auto-labeling categories & priority...')
df['category'] = df.apply(assign_category, axis=1)
df['priority'] = df.apply(assign_priority, axis=1)
df['cat_label'] = df['category'].map(CATEGORY_TO_IDX)
df['pri_label'] = df['priority'].map(PRIORITY_TO_IDX)
df['spam_label'] = df['label']

# Shuffle & split
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
n = len(df)
t1, t2 = int(n*0.80), int(n*0.90)
train_df = df.iloc[:t1].reset_index(drop=True)
val_df   = df.iloc[t1:t2].reset_index(drop=True)
test_df  = df.iloc[t2:].reset_index(drop=True)

print(f'\n✅ Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')
print('\nCategory distribution (train):')
print(train_df['category'].value_counts().to_string())
print('\nPriority distribution (train):')
print(train_df['priority'].value_counts().to_string())

In [ ]:
# ── Cell 6: Model definition ──────────────────────────────────────────────────
import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizerFast

MODEL_NAME = 'distilbert-base-uncased'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {DEVICE}')

class EmailBERT(nn.Module):
    def __init__(self, n_spam=2, n_cat=4, n_pri=3, dropout=0.3, freeze_layers=4):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(MODEL_NAME)
        if freeze_layers > 0:
            for i, layer in enumerate(self.bert.transformer.layer):
                if i < freeze_layers:
                    for p in layer.parameters(): p.requires_grad = False
        h = self.bert.config.hidden_size  # 768
        self.dropout = nn.Dropout(dropout)
        self.spam_head = nn.Sequential(nn.Linear(h,256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256,n_spam))
        self.cat_head  = nn.Sequential(nn.Linear(h,256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256,n_cat))
        self.pri_head  = nn.Sequential(nn.Linear(h,128), nn.GELU(), nn.Dropout(dropout), nn.Linear(128,n_pri))

    def forward(self, input_ids, attention_mask):
        out    = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self.dropout(out.last_hidden_state[:, 0, :])
        return self.spam_head(pooled), self.cat_head(pooled), self.pri_head(pooled)

class MultiTaskLoss(nn.Module):
    def __init__(self, spam_w=2.0, cat_w=1.0, pri_w=1.0):
        super().__init__()
        self.spam_w, self.cat_w, self.pri_w = spam_w, cat_w, pri_w
        self.ce = nn.CrossEntropyLoss()
    def forward(self, sl, s_lbl, cl, c_lbl, pl, p_lbl):
        ls = self.ce(sl, s_lbl)
        lc = self.ce(cl, c_lbl)
        lp = self.ce(pl, p_lbl)
        return self.spam_w*ls + self.cat_w*lc + self.pri_w*lp, ls, lc, lp

tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
print('✅ Model & tokenizer ready')

In [ ]:
# ── Cell 7: Dataset & DataLoader ──────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader

MAX_LEN    = 128
BATCH_SIZE = 32

class EmailDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN):
        self.encodings = tokenizer(
            df['text'].tolist(),
            max_length=max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        self.spam_labels = torch.tensor(df['spam_label'].values, dtype=torch.long)
        self.cat_labels  = torch.tensor(df['cat_label'].values,  dtype=torch.long)
        self.pri_labels  = torch.tensor(df['pri_label'].values,  dtype=torch.long)

    def __len__(self):
        return len(self.spam_labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'spam_label':     self.spam_labels[idx],
            'cat_label':      self.cat_labels[idx],
            'pri_label':      self.pri_labels[idx],
        }

print('Tokenizing datasets (this takes ~2-3 min)...')
train_ds = EmailDataset(train_df, tokenizer)
val_ds   = EmailDataset(val_df,   tokenizer)
test_ds  = EmailDataset(test_df,  tokenizer)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=64,         shuffle=False, num_workers=2, pin_memory=True)
test_dl  = DataLoader(test_ds,  batch_size=64,         shuffle=False, num_workers=2, pin_memory=True)

print(f'✅ Train batches: {len(train_dl)} | Val batches: {len(val_dl)}')

In [ ]:
# ── Cell 8: Training ──────────────────────────────────────────────────────────
import time
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import accuracy_score, f1_score

EPOCHS       = 4
LR           = 2e-5
PATIENCE     = 2
GRAD_CLIP    = 1.0
WARMUP_RATIO = 0.1

model   = EmailBERT().to(DEVICE)
loss_fn = MultiTaskLoss().to(DEVICE)

backbone_params = [p for n,p in model.named_parameters() if 'bert' in n and p.requires_grad]
head_params     = [p for n,p in model.named_parameters() if 'bert' not in n]
optimizer = AdamW([{'params': backbone_params, 'lr': LR},
                   {'params': head_params,     'lr': LR*10}], weight_decay=0.01)

total_steps  = len(train_dl) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
scaler       = torch.cuda.amp.GradScaler()  # mixed precision

def evaluate(model, dl):
    model.eval()
    s_true,s_pred,c_true,c_pred,p_true,p_pred = [],[],[],[],[],[]
    with torch.no_grad():
        for batch in dl:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            sl   = batch['spam_label'].to(DEVICE)
            cl   = batch['cat_label'].to(DEVICE)
            pl   = batch['pri_label'].to(DEVICE)
            s_logit, c_logit, p_logit = model(ids, mask)
            s_true.extend(sl.cpu()); s_pred.extend(s_logit.argmax(1).cpu())
            c_true.extend(cl.cpu()); c_pred.extend(c_logit.argmax(1).cpu())
            p_true.extend(pl.cpu()); p_pred.extend(p_logit.argmax(1).cpu())
    return {
        'spam_acc': accuracy_score(s_true, s_pred),
        'spam_f1':  f1_score(s_true, s_pred, average='binary'),
        'cat_f1':   f1_score(c_true, c_pred, average='weighted'),
        'pri_f1':   f1_score(p_true, p_pred, average='weighted'),
    }

best_f1      = 0.0
patience_cnt = 0
CKPT_PATH    = '/content/best_model.pt'
history      = []

print('='*60)
print('  TRAINING MULTI-TASK BERT')
print('='*60)

for epoch in range(1, EPOCHS+1):
    model.train()
    train_loss = 0.0
    t0 = time.time()

    for step, batch in enumerate(train_dl, 1):
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        sl   = batch['spam_label'].to(DEVICE)
        cl   = batch['cat_label'].to(DEVICE)
        pl   = batch['pri_label'].to(DEVICE)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            s_logit, c_logit, p_logit = model(ids, mask)
            loss, ls, lc, lp = loss_fn(s_logit, sl, c_logit, cl, p_logit, pl)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss += loss.item()

        if step % 100 == 0:
            print(f'  Epoch {epoch} | Step {step}/{len(train_dl)} | loss={loss.item():.4f} '
                  f'(spam={ls.item():.3f} cat={lc.item():.3f} pri={lp.item():.3f})')

    elapsed = time.time() - t0
    metrics = evaluate(model, val_dl)
    composite = 0.6*metrics['spam_f1'] + 0.25*metrics['cat_f1'] + 0.15*metrics['pri_f1']

    print(f'\n{"-"*60}')
    print(f'  Epoch {epoch}/{EPOCHS}  ({elapsed:.0f}s)')
    print(f'  Train loss:   {train_loss/len(train_dl):.4f}')
    print(f'  Spam acc={metrics["spam_acc"]:.4f}  F1={metrics["spam_f1"]:.4f}')
    print(f'  Category F1={metrics["cat_f1"]:.4f}')
    print(f'  Priority F1={metrics["pri_f1"]:.4f}')
    print(f'  Composite score={composite:.4f}')
    history.append({**metrics, 'epoch': epoch, 'composite': composite})

    if composite > best_f1:
        best_f1      = composite
        patience_cnt = 0
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'optimizer': optimizer.state_dict(), 'metrics': metrics}, CKPT_PATH)
        print(f'  ✅ Best model saved! (composite={composite:.4f})')
    else:
        patience_cnt += 1
        print(f'  No improvement ({patience_cnt}/{PATIENCE})')
        if patience_cnt >= PATIENCE:
            print('  Early stopping triggered.')
            break
    print(f'{"-"*60}\n')

print(f'\n🎉 Training complete! Best composite F1: {best_f1:.4f}')

In [ ]:
# ── Cell 9: Final test evaluation ────────────────────────────────────────────
from sklearn.metrics import classification_report

# Load best model
ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])

model.eval()
s_true,s_pred,c_true,c_pred,p_true,p_pred = [],[],[],[],[],[]
with torch.no_grad():
    for batch in test_dl:
        ids  = batch['input_ids'].to(DEVICE)
        mask = batch['attention_mask'].to(DEVICE)
        sl,cl,pl = batch['spam_label'],batch['cat_label'],batch['pri_label']
        s_logit,c_logit,p_logit = model(ids, mask)
        s_true.extend(sl.numpy()); s_pred.extend(s_logit.argmax(1).cpu().numpy())
        c_true.extend(cl.numpy()); c_pred.extend(c_logit.argmax(1).cpu().numpy())
        p_true.extend(pl.numpy()); p_pred.extend(p_logit.argmax(1).cpu().numpy())

print('='*60)
print('  FINAL TEST RESULTS')
print('='*60)
print('\n[SPAM DETECTION]')
print(classification_report(s_true, s_pred, target_names=['ham','spam']))
print('\n[EMAIL CATEGORY]')
print(classification_report(c_true, c_pred, target_names=['personal','promotions','spam','work']))
print('\n[PRIORITY SCORING]')
print(classification_report(p_true, p_pred, target_names=['Low','Medium','High']))

In [ ]:
# ── Cell 10: Quick inference demo ─────────────────────────────────────────────
def predict(text):
    enc = tokenizer(text, max_length=128, padding='max_length',
                    truncation=True, return_tensors='pt')
    ids  = enc['input_ids'].to(DEVICE)
    mask = enc['attention_mask'].to(DEVICE)
    model.eval()
    with torch.no_grad():
        s_logit, c_logit, p_logit = model(ids, mask)
    s_prob = torch.softmax(s_logit, dim=-1)[0]
    c_prob = torch.softmax(c_logit, dim=-1)[0]
    p_prob = torch.softmax(p_logit, dim=-1)[0]
    si, ci, pi = s_prob.argmax().item(), c_prob.argmax().item(), p_prob.argmax().item()
    return {
        'spam':     SPAM_LABELS[si],
        'is_spam':  si == 1,
        'spam_conf': round(s_prob[si].item()*100, 1),
        'category': CATEGORY_LABELS[ci],
        'priority': PRIORITY_LABELS[pi],
    }

samples = [
    "Congratulations! You've won £1,000. Click HERE to claim your prize NOW!",
    "Hi team, the quarterly budget report is due by end of day today. Please respond.",
    "Get 50% off on all items this weekend! Limited offer. Subscribe now.",
    "Hey! Are you free for lunch tomorrow? Let me know.",
    "URGENT: Your account has been compromised. Verify immediately.",
]

print(f"{'Email':<55} {'Spam':^12} {'Category':^12} {'Priority':^10}")
print('-'*92)
for text in samples:
    r = predict(text)
    label = f"{r['spam']} ({r['spam_conf']}%)"
    print(f"{text[:54]:<55} {label:^12} {r['category']:^12} {r['priority']:^10}")

In [ ]:
# ── Cell 11: Save model to Google Drive ───────────────────────────────────────
import shutil, json

SAVE_DIR = f'{MODELS_DIR}/saved_models'
os.makedirs(f'{SAVE_DIR}/tokenizer', exist_ok=True)

# Save checkpoint
shutil.copy(CKPT_PATH, f'{SAVE_DIR}/best_model.pt')

# Save tokenizer
tokenizer.save_pretrained(f'{SAVE_DIR}/tokenizer')

# Save label maps
label_maps = {
    'SPAM_LABELS': SPAM_LABELS,
    'CATEGORY_LABELS': CATEGORY_LABELS,
    'PRIORITY_LABELS': PRIORITY_LABELS,
    'training_history': history,
}
with open(f'{SAVE_DIR}/label_maps.json', 'w') as f:
    json.dump(label_maps, f, indent=2)

print('✅ Model saved to Google Drive!')
print(f'   📁 {SAVE_DIR}/')
print(f'      ├── best_model.pt     ← main model weights')
print(f'      ├── tokenizer/        ← tokenizer files')
print(f'      └── label_maps.json   ← label mappings')
print()
print('📥 Next: Download the `saved_models/` folder and place it at:')
print('   email-management-system/ml_pipeline/saved_models/')

In [ ]:
# ── Cell 12: (Optional) Zip & download directly ───────────────────────────────
# Run this if you want to download the model directly without Google Drive
!zip -r /content/email_models.zip /content/best_model.pt
from google.colab import files
print('Downloading model zip...')
files.download('/content/email_models.zip')
print('✅ Download started!')